## Final Workflow: ML based integration pipeline (Sorted Neighbourhood Blocker)

In [104]:
from pathlib import Path
import pandas as pd
from PyDI.io import load_parquet
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 25)
pd.set_option('display.max_colwidth', 100)

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

PIPELINE_DIR = OUTPUT_DIR / "data_fusion"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

In [105]:
amazon_sample = load_parquet(
    DATA_DIR / "amazon_sample.parquet",
    name="amazon_sample"
)

goodreads_sample = load_parquet(
    DATA_DIR / "goodreads_sample.parquet",
    name="goodreads_sample"
)

metabooks_sample = load_parquet(
  DATA_DIR / "metabooks_sample.parquet",
  name="metabooks_sample"
)

## Entity Matching

In [106]:
from PyDI.io import load_csv

train_m2a = load_csv(
    MLDS_DIR / "train_MA.csv",
    name="train_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2a = load_csv(
    MLDS_DIR / "test_MA.csv",
    name="test_metabooks_amazon",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

train_m2g = load_csv(
    MLDS_DIR / "train_MG.csv",
    name="train_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

test_m2g = load_csv(
    MLDS_DIR / "test_MG.csv",
    name="test_metabooks_goodreads",
    header=0,
    names=['id1', 'id2', 'label'],
    add_index=False
)

In [107]:
from PyDI.entitymatching import StringComparator, NumericComparator

comparators = [
    StringComparator(column='title',similarity_function='jaccard'),
    StringComparator(column='title',similarity_function='cosine'),
    StringComparator(column='title',similarity_function='levenshtein'),
    StringComparator(column='title',similarity_function='jaro_winkler'),
    
    StringComparator(column='author',similarity_function='jaccard', preprocess=str.lower),
    StringComparator(column='author',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='author',similarity_function='levenshtein', preprocess=str.lower),

    StringComparator(column='publisher',similarity_function='jaccard', preprocess=str.lower),
    StringComparator(column='publisher',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='publisher',similarity_function='levenshtein', preprocess=str.lower),
    

    NumericComparator(column='publish_year',max_difference=1),

    StringComparator(column='genres',
                     similarity_function='jaccard',
                     preprocess=str.lower, 
                     list_strategy='concatenate'),

    NumericComparator(column="price", max_difference=5),
    NumericComparator(column="page_count", max_difference=10),
    NumericComparator(column="numratings", max_difference=100)
]

In [108]:
from PyDI.entitymatching import SortedNeighbourhoodBlocker


blocker_m2a = SortedNeighbourhoodBlocker(
    metabooks_sample, amazon_sample,
    key='title',
    window=10,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)



blocker_m2g = SortedNeighbourhoodBlocker(
    metabooks_sample, goodreads_sample,
    key='title',
    window=10,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

sorted_candidates_m2a = blocker_m2a.materialize()
sorted_candidates_m2g = blocker_m2g.materialize()

### Evaluate Blocking

In [109]:
from PyDI.entitymatching import EntityMatchingEvaluator


# evaluate blocking: metabooks -> amazon
results_m2a = EntityMatchingEvaluator.evaluate_blocking(
    candidate_pairs=sorted_candidates_m2a,
    blocker=blocker_m2a,
    test_pairs=train_m2a,
    out_dir=BLOCK_EVAL_DIR
)


display(results_m2a)

{'pair_completeness': 0.9456521739130435,
 'pair_quality': 0.0004812119904753213,
 'reduction_ratio': 0.9997048269387755,
 'total_candidates': 361587,
 'total_possible_pairs': 1225000000,
 'true_positives_found': 174,
 'total_true_pairs': 184,
 'evaluation_timestamp': '2025-11-12T17:12:47.779536',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_detailed_results.csv']}

In [110]:
# evaluate blocking: metabooks -> goodreads
results_m2g = EntityMatchingEvaluator.evaluate_blocking(
    candidate_pairs=sorted_candidates_m2g,
    blocker=blocker_m2g,
    test_pairs=train_m2g,
    out_dir=BLOCK_EVAL_DIR
)
display(results_m2g)

{'pair_completeness': 0.9125,
 'pair_quality': 0.0004306899003218375,
 'reduction_ratio': 0.9997232726530613,
 'total_candidates': 338991,
 'total_possible_pairs': 1225000000,
 'true_positives_found': 146,
 'total_true_pairs': 160,
 'evaluation_timestamp': '2025-11-12T17:12:57.751119',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/blocking_detailed_results.csv']}

## ML-based Matcher

In [111]:
from PyDI.entitymatching import FeatureExtractor

feature_extractor = FeatureExtractor(comparators)

train_features_m2a = feature_extractor.create_features(
    metabooks_sample, amazon_sample, train_m2a[['id1', 'id2']], labels=train_m2a['label'], id_column='id'
)

train_features_m2g = feature_extractor.create_features(
    metabooks_sample, goodreads_sample, train_m2g[['id1', 'id2']], labels=train_m2g['label'], id_column='id'
)

feature_columns_m2a = [col for col in train_features_m2a.columns if col not in ['id1', 'id2', 'label']]
X_train_m2a = train_features_m2a[feature_columns_m2a]
y_train_m2a = train_features_m2a['label']

feature_columns_m2g = [col for col in train_features_m2g.columns if col not in ['id1', 'id2', 'label']]
X_train_m2g = train_features_m2g[feature_columns_m2g]
y_train_m2g = train_features_m2g['label']

training_datasets = [(X_train_m2a, y_train_m2a),(X_train_m2g, y_train_m2g)]

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score
import pandas as pd

# classifiers
classifiers = {
    'RandomForestClassifier': RandomForestClassifier(random_state=42),
    'GradientBoostingClassifier': GradientBoostingClassifier(random_state=42),
    'SVC': SVC(probability=True, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42)
}

# Define parameter grids
param_grids = {
    'RandomForestClassifier': {
        'n_estimators': [50, 100, 200,500],
        'max_depth': [None, 10, 20],
        'class_weight': ['balanced', None],
        'min_samples_split': [2, 5]
    },
    'GradientBoostingClassifier': {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5]
    },
    'SVC': {
        'C': [0.1, 1, 10],
        'kernel': ['linear', 'rbf'],
        'gamma': ['scale', 'auto']
    },
    'LogisticRegression': {
        'C': [0.1, 1, 10],
        'penalty': ['l1','l2'],
        'solver': ['lbfgs', 'liblinear']
    }
}


scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_models = []


for data in training_datasets:
    grid_search_results = {}
    best_overall_score = -1
    best_overall_model = None
    best_model_name = None

    for name, model in classifiers.items():
        print(f"Running GridSearchCV for {name}...")
        
        grid = GridSearchCV(
            estimator=model,
            param_grid=param_grids[name],
            scoring=scorer,
            cv=cv_folds,
            n_jobs=-1,
            verbose=0
        )
        
        grid.fit(data[0], data[1])
        
        grid_search_results[name] = {
            'grid_search': grid,
            'best_score': grid.best_score_,
            'best_params': grid.best_params_,
            'best_estimator': grid.best_estimator_
        }

        if grid.best_score_ > best_overall_score:
            best_overall_model = grid.best_estimator_

    best_models.append(best_overall_model)

Running GridSearchCV for RandomForestClassifier...


Running GridSearchCV for GradientBoostingClassifier...
Running GridSearchCV for SVC...
Running GridSearchCV for LogisticRegression...
Running GridSearchCV for RandomForestClassifier...
Running GridSearchCV for GradientBoostingClassifier...
Running GridSearchCV for SVC...
Running GridSearchCV for LogisticRegression...


In [114]:
best_models

[LogisticRegression(C=10, max_iter=1000, random_state=42, solver='liblinear'),
 LogisticRegression(C=10, max_iter=1000, random_state=42, solver='liblinear')]

In [116]:
from PyDI.entitymatching import MLBasedMatcher


ml_matcher = MLBasedMatcher(feature_extractor)

correspondences_m2a = ml_matcher.match(
    metabooks_sample, amazon_sample,
    candidates=blocker_m2a,
    id_column='id',
    trained_classifier=best_models[0],
)

correspondences_m2g = ml_matcher.match(
    metabooks_sample, goodreads_sample,
    candidates=blocker_m2g,
    id_column='id',
    trained_classifier=best_models[1]
)

In [117]:
correspondences_m2a.shape

(30542, 4)

In [118]:
correspondences_m2a.to_csv(CORR_DIR / "workflow_corr_m2a.csv", index=False)
correspondences_m2g.to_csv(CORR_DIR / "workflow_corr_m2g.csv", index=False)

### Evaluate Matching

In [119]:
debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

# evaluate matching metabooks -> amazon
eval_results_m2a = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2a,
    test_pairs=test_m2a,
    out_dir=debug_output_dir
)

display(eval_results_m2a)

{'precision': 0.9459459459459459,
 'recall': 0.7608695652173914,
 'f1': 0.8433734939759038,
 'accuracy': 0.935,
 'true_positives': 35,
 'false_positives': 2,
 'false_negatives': 11,
 'true_negatives': 152,
 'threshold_used': 0.0,
 'total_correspondences': 30542,
 'filtered_correspondences': 30542,
 'evaluation_timestamp': '2025-11-12T17:18:53.101572',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/debug_results_entity_matching/matching_detailed_results.csv']}

In [120]:
# evaluate matching metabooks -> goodreads
eval_results_m2g = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2g,
    test_pairs=test_m2g,
    out_dir=debug_output_dir
)

display(eval_results_m2g)

{'precision': 1.0,
 'recall': 0.775,
 'f1': 0.8732394366197184,
 'accuracy': 0.955,
 'true_positives': 31,
 'false_positives': 0,
 'false_negatives': 9,
 'true_negatives': 160,
 'threshold_used': 0.0,
 'total_correspondences': 8797,
 'filtered_correspondences': 8797,
 'evaluation_timestamp': '2025-11-12T17:18:53.974664',
 'output_files': ['/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/debug_results_entity_matching/matching_detailed_results.csv']}

### Cluster Analysis

In [124]:
cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
cluster_analysis_dir.mkdir(parents=True, exist_ok=True)
cluster_distribution_m2a = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2a,
    out_dir=cluster_analysis_dir
)
cluster_distribution_m2g = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2g,
    out_dir=cluster_analysis_dir
)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Amazon):")
display(cluster_distribution_m2a)
print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Goodreads):")
display(cluster_distribution_m2g)


📊 Cluster Size Distribution Results (Metabooks -> Amazon):


,cluster_size,frequency,percentage
0,2,19366,87.807753
1,3,959,4.348220
2,4,1117,5.064611
3,5,225,1.020177
4,6,160,0.725459
5,7,74,0.335525
6,8,58,0.262979
7,9,27,0.122421
8,10,23,0.104285
9,11,12,0.054409



📊 Cluster Size Distribution Results (Metabooks -> Goodreads):


,cluster_size,frequency,percentage
0,2,5685,84.184807
1,3,726,10.750777
2,4,186,2.754331
3,5,77,1.140234
4,6,39,0.577521
5,7,22,0.325781
6,8,8,0.118466
7,9,2,0.029616
8,10,3,0.044425
9,11,2,0.029616


In [125]:
from PyDI.entitymatching import MaximumBipartiteMatching, StableMatching

# We are using Maxmimum Bipartite Matching to refine results to 1:1 matches
clusterer = MaximumBipartiteMatching()
correspondences_m2a = clusterer.cluster(correspondences_m2a)
correspondences_m2g = clusterer.cluster(correspondences_m2g)
correspondences_m2a.to_csv(CORR_DIR / "workflow_corr_m2a.csv", index=False)
correspondences_m2g.to_csv(CORR_DIR / "workflow_corr_m2g.csv", index=False)

In [126]:
cluster_analysis_dir = OUTPUT_DIR / "cluster_analysis"
cluster_analysis_dir.mkdir(parents=True, exist_ok=True)

cluster_distribution_m2a = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2a,
    out_dir=cluster_analysis_dir
)
cluster_distribution_m2g = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2g,
    out_dir=cluster_analysis_dir
)

print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Goodreads):")
display(cluster_distribution_m2g)

print(f"\n📊 Cluster Size Distribution Results (Metabooks -> Amazon):")
display(cluster_distribution_m2a)


📊 Cluster Size Distribution Results (Metabooks -> Goodreads):


,cluster_size,frequency,percentage
0,2,7007,100.0



📊 Cluster Size Distribution Results (Metabooks -> Amazon):


,cluster_size,frequency,percentage
0,2,24375,100.0


## Data Fusion

In [127]:
mbm_correspondences_m2a = pd.read_csv(CORR_DIR / "workflow_corr_m2a.csv")
mbm_correspondences_m2g = pd.read_csv(CORR_DIR / "workflow_corr_m2g.csv")

all_correspondences = pd.concat([mbm_correspondences_m2a, mbm_correspondences_m2g], ignore_index=True)

len(all_correspondences)

31382

In [128]:
from PyDI.fusion import DataFusionStrategy, DataFusionEngine, longest_string, union, prefer_higher_trust
import pandas as pd

metabooks_sample.attrs["trust_score"] = 3
goodreads_sample.attrs["trust_score"] = 2
amazon_sample.attrs["trust_score"] = 1
strategy = DataFusionStrategy('books_fusion_strategy')

strategy.add_attribute_fuser('title', longest_string)
strategy.add_attribute_fuser('author', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('publish_year', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('publisher', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('language', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('price', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('page_count', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('numrating', prefer_higher_trust, trust_key="trust_score")
strategy.add_attribute_fuser('genres',union)

In [129]:
engine = DataFusionEngine(strategy, debug=True, debug_format='json',
                          debug_file= PIPELINE_DIR / "debug_fusion_ml_sn_blocker.jsonl")

fused_ml_snblocker = engine.run(
    datasets=[amazon_sample, metabooks_sample, goodreads_sample],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False
)

print(f'Fused rows: {len(fused_ml_snblocker):,}')

Fused rows: 28,250


In [130]:
fused_ml_snblocker.to_parquet(PIPELINE_DIR / "fused_ml_snblocker.parquet")

### Fusion Evaluation

In [131]:
fused_dataset = pd.read_parquet(PIPELINE_DIR / "fused_ml_snblocker.parquet")
fused_dataset["publish_year"] = fused_dataset["publish_year"].astype("Int16")
fused_dataset["page_count"] = fused_dataset["page_count"].astype("Int32")
golden_fused_dataset= pd.read_parquet(MLDS_DIR / "golden_fused_books.parquet")

In [132]:
from PyDI.fusion import tokenized_match, boolean_match,numeric_tolerance_match,set_equality_match

import numpy as np
import re, ast, numpy as np, pandas as pd


def categories_set_equal(a, b) -> bool:
    """Return True if a and b contain the same unique categories (order/type agnostic)."""
    def to_set(x):
        def items(v):
            # missing
            if v is None or (isinstance(v, float) and np.isnan(v)): return []
            # numpy array → recurse over elements
            if isinstance(v, np.ndarray): 
                out=[]; [out.extend(items(e)) for e in v.flatten()]; return out
            # python containers → recurse over elements
            if isinstance(v, (list, tuple, set)):
                out=[]; [out.extend(items(e)) for e in v]; return out
            # scalar/string: try parse stringified list; else split by delimiters
            s = str(v).strip()
            if s == "" or s.lower() in {"nan","none"}: return []
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)): return items(parsed)
            except Exception:
                pass
            return [p.strip() for p in re.split(r"[|,;/]", s) if p.strip()]
        return {it.lower() for it in items(x)}
    return to_set(a) == to_set(b)

strategy.add_evaluation_function("title", tokenized_match)
strategy.add_evaluation_function("author", tokenized_match)
strategy.add_evaluation_function("publisher", tokenized_match)
strategy.add_evaluation_function("publish_year", numeric_tolerance_match)
strategy.add_evaluation_function("numratings", numeric_tolerance_match)
strategy.add_evaluation_function("price", numeric_tolerance_match)
strategy.add_evaluation_function("page_count", numeric_tolerance_match)
strategy.add_evaluation_function("language", tokenized_match)
strategy.add_evaluation_function("genres", categories_set_equal)

In [133]:
from PyDI.fusion import DataFusionEvaluator
fused_dataset.drop_duplicates(subset='isbn_clean', keep='first',inplace=True)
# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused_dataset,
    fused_id_column='isbn_clean',
    gold_df=golden_fused_dataset,
    gold_id_column='isbn_clean',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

Evaluating fusion results against gold standard...

Fusion Evaluation Results:
  overall_accuracy: 0.658
  macro_accuracy: 0.658
  num_evaluated_records: 38828
  num_evaluated_attributes: 9
  total_evaluations: 342014
  total_correct: 225138
  title_accuracy: 0.671
  title_count: 38828
  price_accuracy: 0.662
  price_count: 36440
  genres_accuracy: 0.636
  genres_count: 37448
  author_accuracy: 0.643
  author_count: 38828
  language_accuracy: 0.698
  language_count: 38695
  numratings_accuracy: 0.614
  numratings_count: 38828
  publisher_accuracy: 0.674
  publisher_count: 38828
  page_count_accuracy: 0.665
  page_count_count: 35411
  publish_year_accuracy: 0.660
  publish_year_count: 38708

Overall Accuracy: 65.8%


In [134]:
golden_fused_dataset[golden_fused_dataset.isbn_clean.isin(isbns)].sort_values(by='isbn_clean')["genres"].iloc[0]

NameError: name 'isbns' is not defined

In [137]:
fused_dataset[fused_dataset.isbn_clean.isin(isbns)].sort_values(by='isbn_clean')["genres"].iloc[2]

array(['',
       'Historical Fiction, Fiction, Historical, Medieval, British Literature, Plantagenet, 12th Century, France, Medieval History, Novels'],
      dtype=object)

In [144]:
import numpy as np
import re
categories_set_equal(fused_dataset[fused_dataset.isbn_clean.isin(isbns)].sort_values(by='isbn_clean')["genres"].iloc[7],golden_fused_dataset[golden_fused_dataset.isbn_clean.isin(isbns)].sort_values(by='isbn_clean')["genres"].iloc[7])

True

In [147]:
fused_dataset.head()

,_id,_fusion_group_id,_fusion_sources,rating,id,isbn_clean,title,bookformat,genres,price,author,numratings,language,publisher,edition,page_count,publish_year,_fusion_confidence,_fusion_metadata
0,goodreads_36823,group_0,"[metabooks_sample, goodreads_sample]",4.37,goodreads_36823,0756408105,once broken faith october daye,Mass Market Paperback,"[Literature, Fiction, Genre Fiction, Urban Fantasy, Fantasy, Fae, Paranormal, Fiction, Magic, Ro...",8.99,seanan mcguire,7921,English,DAW,nan,432,2016,0.785897,"{'_id_inputs': [{'dataset': 'goodreads_sample', 'record_id': 'goodreads_36823', 'value': 'goodre..."
1,amazon_73825,group_1,"[amazon_sample, metabooks_sample]",4.20,amazon_73825,0373036639,provocative proposal wedded blitz romance 3663,None,"[Romance, Contemporary]",7.48,day leclaire,2,English,Harlequin,None,192,2001,0.692308,"{'_id_inputs': [{'dataset': 'amazon_sample', 'record_id': 'amazon_73825', 'value': 'amazon_73825..."
2,metabooks_479861,group_2,"[amazon_sample, goodreads_sample, metabooks_sample]",4.30,metabooks_479861,1860498345,good behavior,Paperback,"[Literature, Fiction, Genre Fiction, New Adult, Romance, Contemporary, High School, Young Adult,...",8.62,molly keane,1219,English,Virago Pr,nan,245,1981,0.769231,"{'_id_inputs': [{'dataset': 'metabooks_sample', 'record_id': 'metabooks_479861', 'value': 'metab..."
3,amazon_214015,group_3,"[amazon_sample, metabooks_sample]",4.70,amazon_214015,0316604429,newcombs wildflower guide,None,"[Science, Math, Biological Sciences]",12.69,lawrence newcomb,872,English,"Little, Brown and Company",None,490,1989,0.692308,"{'_id_inputs': [{'dataset': 'amazon_sample', 'record_id': 'amazon_214015', 'value': 'amazon_2140..."
4,amazon_25684,group_4,"[amazon_sample, metabooks_sample]",4.70,amazon_25684,048621866X,the egyptian book of the dead the papyrus of ani in the british museum,None,"[History, Americas]",12.99,matthew vossler,629,English,Dover Publications,None,377,1967,0.692308,"{'_id_inputs': [{'dataset': 'amazon_sample', 'record_id': 'amazon_25684', 'value': 'amazon_25684..."


In [148]:
isbns=list(set(fused_dataset.isbn_clean)&set(golden_fused_dataset.isbn_clean))
display(fused_dataset[fused_dataset.isbn_clean.isin(isbns)].sort_values(by='isbn_clean')[["isbn_clean","genres","_id"]].head(10))
display(golden_fused_dataset[golden_fused_dataset.isbn_clean.isin(isbns)].sort_values(by='isbn_clean')[["isbn_clean","genres"]].head(10))

,isbn_clean,genres,_id
25645,0000913154,"[Engineering, Transportation]",metabooks_279855
19963,000100039X,"[Literature, Fiction, Poetry, Poetry, Philosophy, Classics, Fiction, Spirituality, Religion, Lit...",metabooks_19041
7078,0002118572,"[, Historical Fiction, Fiction, Historical, Medieval, British Literature, Plantagenet, 12th Cent...",amazon_203216
17268,0002160595,"[Biographies, Memoirs, Leaders, Notable People]",amazon_33718
21984,0002167425,"[Science, Math, Biological Sciences]",amazon_254251
18633,0002176084,"[History, Europe]",metabooks_264544
18210,0002213958,"[Historical Fiction, Fiction, World War II, Romance, Classics, Historical, Literature, Fiction, ...",metabooks_520677
8121,0002219476,"[Mystery, Thriller, Suspense, Thrillers]",metabooks_195282
3391,0002251892,"[Sports, Outdoors, Biographies]",metabooks_79754
13972,000225414X,"[Literature, Fiction, Genre Fiction]",amazon_45395


,isbn_clean,genres
0,0000913154,"Engineering, Transportation"
1,000100039X,"Classics, Fiction, Inspirational, Literature, Novels, Philosophy, Poetry, Religion, Self Help, S..."
5,0002118572,None
6,0002160595,"Biographies, Leaders, Memoirs, Notable People"
7,0002167425,"Biological Sciences, Math, Science"
8,0002176084,"Europe, History"
9,0002213958,"Classics, Contemporary, Fiction, Historical, Historical Fiction, Literature, Romance, World War Ii"
10,0002219476,"Mystery, Thriller, Suspense, Thrillers"
15,0002251892,"Biographies, Outdoors, Sports"
16,000225414X,"Fiction, Genre Fiction, Literature"


In [180]:
metabooks[metabooks.isbn_clean=="0002213958"]

,id,title,author_name,rating,num_rating,language,genres,publisher,page_count,price,publish_year,isbn_clean
520676,metabooks_520677,a tender victory,taylor caldwell,4.3,618,English,"['Literature', 'Fiction', 'Contemporary']",HarperCollins Publishers,<NA>,9.53,1988,0002213958


In [25]:
categories_set_equal(fused_dataset[fused_dataset.isbn_clean.isin(isbns)].sort_values(by='isbn_clean')["genres"].iloc[1],
                     golden_fused_dataset[golden_fused_dataset.isbn_clean.isin(isbns)].sort_values(by='isbn_clean')["genres"].iloc[1])



True

In [65]:
golden_fused_dataset[golden_fused_dataset.isbn_clean.isin(isbns)].sort_values(by='isbn_clean')["genres"].iloc[55]


'Contemporary, Fiction, Literature'

In [64]:
fused_dataset[fused_dataset.isbn_clean.isin(isbns)].sort_values(by='isbn_clean')[["genres","isbn_clean"]].iloc[55]

genres        [['Literature', 'Fiction', 'Contemporary']]
isbn_clean                                     0006550835
Name: 1548, dtype: object

In [67]:
metabooks_sample[metabooks_sample.isbn_clean=="0006550835"]

,id,title,author,rating,numratings,language,genres,bookformat,edition,page_count,publisher,publish_year,price,isbn_clean,has_nested_genres
15918,metabooks_259863,mara and dann,doris lessing,4.2,119,English,"['Literature', 'Fiction', 'Contemporary']",None,None,416,Flamingo,<NA>,10.39,0006550835,False
